In [ ]:
!nvidia-smi

In [ ]:
import gc
import hashlib
import json
import os
import time
import uuid
import atexit
from pathlib import Path

import gradio as gr

import core_config
from core_config import SYSTEM_DIRS
from oec_gpu_admission import select_idle_a100_gpu
from storage_manager import managed_storage_operation, storage_preflight
from storage_policy import BYTES_PER_GB, estimate_hf_repo_bytes
from tools.media_services.model_cache import RuntimeLock

# Select an admitted physical A100 before importing torch or Diffusers.
_GPU_SELECTION = select_idle_a100_gpu(caller="WAN_VIDEO_NOTEBOOK")
_PHYSICAL_GPU = str(_GPU_SELECTION["index"])
os.environ["CUDA_VISIBLE_DEVICES"] = _PHYSICAL_GPU
os.environ["OEC_WAN_PHYSICAL_GPU"] = _PHYSICAL_GPU
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")

import torch
# Diffusers probes optional XPU APIs during import even when this notebook runs
# CUDA inference. Repair only missing hooks on an unavailable XPU backend.
from tools.wan_xpu_compat import ensure_torch_xpu_compat
ensure_torch_xpu_compat(torch)

from huggingface_hub import snapshot_download
from huggingface_hub.errors import LocalEntryNotFoundError
from diffusers import AutoencoderKLWan, WanPipeline
from diffusers.utils import export_to_video

MODEL_ID = "Wan-AI/Wan2.1-T2V-14B-Diffusers"
CACHE_DIR = Path(SYSTEM_DIRS["models"]).expanduser().resolve() / "hub"
DOWNLOAD_ALLOWED = os.environ.get("OEC_WAN_ALLOW_DOWNLOAD", "0") == "1"
SMOKE_MODE = os.environ.get("OEC_WAN_HEADLESS_SMOKE", "0") == "1"
OUTPUT_DIR = Path(
    os.environ.get(
        "OEC_WAN_OUTPUT_DIR",
        str(Path(SYSTEM_DIRS["outputs"]).expanduser().resolve() / "videos"),
    )
).expanduser().resolve()


def _wan_snapshot():
    try:
        path = snapshot_download(
            repo_id=MODEL_ID, cache_dir=str(CACHE_DIR), local_files_only=True,
        )
        print("WAN_CACHE=HIT", flush=True)
        return Path(path)
    except LocalEntryNotFoundError as cache_error:
        if not DOWNLOAD_ALLOWED:
            raise RuntimeError(
                "Wan 2.1 cache miss. This notebook is cache-only by default; "
                "set OEC_WAN_ALLOW_DOWNLOAD=1 only for an explicitly admitted test run."
            ) from cache_error

    # The official Wan 2.1 T2V 14B snapshot is roughly 80 GB; never start a
    # partial download on an optimistic/default byte estimate.
    estimated_bytes = int(estimate_hf_repo_bytes(MODEL_ID) or 0)
    required_gb = max(85.0, estimated_bytes / BYTES_PER_GB + 2.0)
    minimum_free_gb = max(2.0, float(os.environ.get("AIEROS_MODEL_MIN_FREE_GB", "2.0")))
    ok, reason = storage_preflight(
        "models", required_gb=required_gb,
        require_fresh_quota=True, minimum_free_gb=minimum_free_gb,
    )
    print(f"WAN_STORAGE_ADMISSION={bool(ok)} required_gb={required_gb:.3f} reason={reason}", flush=True)
    if not ok:
        raise RuntimeError(f"StorageManager denied Wan model download: {reason}")
    os.environ["HF_HUB_OFFLINE"] = "0"
    os.environ["TRANSFORMERS_OFFLINE"] = "0"
    with managed_storage_operation(
        "models", required_gb=required_gb,
        require_fresh_quota=True, minimum_free_gb=minimum_free_gb,
    ):
        ok, reason = storage_preflight(
            "models", required_gb=required_gb,
            require_fresh_quota=True, minimum_free_gb=minimum_free_gb,
        )
        if not ok:
            raise RuntimeError(f"StorageManager denied Wan download inside managed operation: {reason}")
        path = snapshot_download(
            repo_id=MODEL_ID, cache_dir=str(CACHE_DIR), local_files_only=False,
        )
    print(f"WAN_DOWNLOAD=PASS snapshot={path}", flush=True)
    return Path(path)


class WanVideoWorker:
    def __init__(self):
        self.gpu_id = _PHYSICAL_GPU
        self.pipe = None
        self.vae = None
        self.snapshot = None
        self.runtime_lock = RuntimeLock("wan-t2v-14b")
        self.lock_owned = False
        self.last_output_path = None

    def load_model(self):
        if self.pipe is not None:
            return
        if not torch.cuda.is_available():
            raise RuntimeError("CUDA is unavailable; Wan inference requires an admitted A100")
        if "A100" not in torch.cuda.get_device_name(0).upper():
            raise RuntimeError("Selected visible device is not an A100")
        self.runtime_lock.acquire()
        self.lock_owned = True
        try:
            self.snapshot = _wan_snapshot()
            self.vae = AutoencoderKLWan.from_pretrained(
                str(self.snapshot), subfolder="vae",
                torch_dtype=torch.float32, local_files_only=True,
            )
            self.pipe = WanPipeline.from_pretrained(
                str(self.snapshot), vae=self.vae,
                torch_dtype=torch.bfloat16, local_files_only=True,
            )
            self.pipe.enable_model_cpu_offload()
            print(f"WAN_MODEL_READY=PASS physical_gpu={self.gpu_id} snapshot={self.snapshot}", flush=True)
        except Exception:
            self.close()
            raise

    def generate(self, prompt, num_frames=None, fps=16):
        self.load_model()
        if num_frames is None:
            num_frames = int(os.environ.get("OEC_WAN_NUM_FRAMES", "9" if SMOKE_MODE else "81"))
        if isinstance(num_frames, bool) or int(num_frames) < 5 or (int(num_frames) - 1) % 4:
            raise ValueError("num_frames must be >=5 and satisfy (num_frames - 1) % 4 == 0")
        num_frames, fps = int(num_frames), int(fps)
        if not 8 <= fps <= 30:
            raise ValueError("fps must be between 8 and 30")
        if not isinstance(prompt, str) or not prompt.strip():
            raise ValueError("prompt must be non-empty")
        print(f"WAN_GENERATION_START frames={num_frames} fps={fps}", flush=True)
        frames = self.pipe(
            prompt=prompt.strip(), num_frames=num_frames, height=480, width=832,
            guidance_scale=5.0,
        ).frames[0]

        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        output_path = OUTPUT_DIR / f"wan-{uuid.uuid4().hex}.mp4"
        if SMOKE_MODE:
            # Acceptance output is run-scoped under /tmp; model downloads remain guarded.
            export_to_video(frames, str(output_path), fps=fps)
        else:
            estimated_gb = max(0.05, len(frames) * 0.10 / BYTES_PER_GB)
            with managed_storage_operation(
                "outputs", required_gb=estimated_gb,
                require_fresh_quota=True, minimum_free_gb=0.05,
            ):
                export_to_video(frames, str(output_path), fps=fps)
        if not output_path.is_file() or output_path.stat().st_size <= 1024:
            output_path.unlink(missing_ok=True)
            raise RuntimeError("Wan returned an empty or incomplete MP4")
        self.last_output_path = output_path
        digest = hashlib.sha256(output_path.read_bytes()).hexdigest()
        print("WAN_ASSET=" + json.dumps({
            "path": str(output_path), "bytes": output_path.stat().st_size,
            "sha256": digest, "frames": len(frames), "fps": fps,
            "duration_seconds": round(len(frames) / fps, 4),
            "physical_gpu": self.gpu_id, "model_id": MODEL_ID, "snapshot": str(self.snapshot),
        }, sort_keys=True), flush=True)
        return output_path

    def close(self):
        self.pipe = None
        self.vae = None
        gc.collect()
        try:
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception:
            pass
        if self.lock_owned:
            self.runtime_lock.release()
            self.lock_owned = False


worker = WanVideoWorker()


def generate_wan_video(prompt, num_frames, fps):
    """Generate a real MP4 from the Gradio UI and return its path plus metadata."""
    output_path = worker.generate(
        prompt, num_frames=int(num_frames), fps=int(fps),
    )
    metadata = {
        "status": "PASS",
        "path": str(output_path),
        "bytes": output_path.stat().st_size,
        "sha256": hashlib.sha256(output_path.read_bytes()).hexdigest(),
        "frames": int(num_frames),
        "fps": int(fps),
        "duration_seconds": round(int(num_frames) / int(fps), 4),
        "model_id": MODEL_ID,
        "physical_gpu": _PHYSICAL_GPU,
    }
    return str(output_path), json.dumps(metadata, indent=2, ensure_ascii=False)


demo = gr.Interface(
    fn=generate_wan_video,
    inputs=[
        gr.Textbox(
            label="Video prompt", lines=3,
            value="A short cinematic camera move across a glowing bioluminescent forest at night",
        ),
        gr.Slider(5, 81, value=17, step=4, label="Frames (must satisfy (n - 1) % 4 == 0)"),
        gr.Slider(8, 30, value=16, step=1, label="Frames per second"),
    ],
    outputs=[
        gr.Video(label="Generated MP4"),
        gr.Textbox(label="Asset metadata", lines=10),
    ],
    title="Wan 2.1 T2V 14B — Gradio video generator",
    description="Generates a real MP4 using the admitted A100. The UI binds only to loopback; model weights are loaded on the first Generate click.",
    allow_flagging="never",
)

if SMOKE_MODE:
    # Headless acceptance still exercises the real worker and validates its MP4.
    try:
        WAN_OUTPUT_PATH = worker.generate(
            os.environ.get(
                "OEC_WAN_SMOKE_PROMPT",
                "A short cinematic camera move across a glowing bioluminescent forest at night",
            ),
            num_frames=int(os.environ.get("OEC_WAN_NUM_FRAMES", "9")),
        )
    finally:
        worker.close()
else:
    # Keep weights resident for later Gradio callbacks; release them on kernel exit.
    atexit.register(worker.close)
    demo.queue(max_size=1).launch(
        server_name="127.0.0.1",
        server_port=7909,
        share=False,
        inline=False,
        show_error=True,
        allowed_paths=[str(OUTPUT_DIR)],
    )
